In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor


# ============================================================
# 1. Tentukan lokasi dataset dan folder output
# ============================================================

nama_dataset = "dataset_fitur_jumlah_sel.csv"
folder_kaggle_input = Path("/kaggle/input")

if folder_kaggle_input.is_dir():
    kandidat_dataset = list(folder_kaggle_input.rglob(nama_dataset))

    if not kandidat_dataset:
        raise FileNotFoundError(
            f"{nama_dataset} tidak ditemukan di {folder_kaggle_input}"
        )
    if len(kandidat_dataset) > 1:
        raise RuntimeError(
            "Ditemukan lebih dari satu dataset. Tentukan path secara eksplisit:\n"
            + "\n".join(str(path) for path in kandidat_dataset)
        )

    file_data = kandidat_dataset[0]
else:
    file_data = Path(
        r"D:\00 Kuliah\TA\Experiment\Model\5 ML\dataset_fitur_jumlah_sel.csv"
    )

if not file_data.is_file():
    raise FileNotFoundError(f"File dataset tidak ditemukan: {file_data}")

if Path("/kaggle/working").is_dir():
    folder_output = Path("/kaggle/working") / "Hasil" / "Random"
else:
    folder_output = file_data.parent / "Hasil" / "Random"

folder_output.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {file_data}")
print(f"Folder output: {folder_output}")


# ============================================================
# 2. Baca dan validasi dataset
# ============================================================

kolom_nama = "nama_file"
kolom_target = "Jumlah Sel (sel/mL)"
SEED = 42
TEST_RATIO = 0.20

kolom = pd.read_csv(
    file_data,
    encoding="utf-8-sig",
    nrows=0,
).columns.tolist()

kolom_fitur = [nama for nama in kolom if nama.startswith("fitur_")]
kolom_wajib = {kolom_nama, kolom_target}

if not kolom_wajib.issubset(kolom):
    kolom_hilang = sorted(kolom_wajib - set(kolom))
    raise KeyError(f"Kolom wajib tidak ditemukan: {kolom_hilang}")

if not kolom_fitur:
    raise ValueError("Tidak ditemukan kolom fitur dengan awalan 'fitur_'.")

# Baca fitur sebagai float32 untuk mengurangi penggunaan memori.
dtype_kolom = {nama: np.float32 for nama in kolom_fitur}
dtype_kolom[kolom_nama] = "string"
dtype_kolom[kolom_target] = "string"

df = pd.read_csv(
    file_data,
    encoding="utf-8-sig",
    dtype=dtype_kolom,
)

# Hilangkan titik pemisah ribuan pada target, misalnya "121.847.700".
target_teks = df[kolom_target].astype("string").str.strip()
format_ribuan = target_teks.str.fullmatch(
    r"\d{1,3}(?:\.\d{3})+"
).fillna(False)

target_teks.loc[format_ribuan] = target_teks.loc[
    format_ribuan
].str.replace(".", "", regex=False)

y = pd.to_numeric(
    target_teks,
    errors="coerce",
).to_numpy(dtype=np.float64)

# Hanya kolom fitur_... yang menjadi input model.
# Kolom nama_file dan target tidak dimasukkan ke X.
X = df[kolom_fitur].to_numpy(dtype=np.float32, copy=True)

if df[kolom_nama].isna().any() or df[kolom_nama].duplicated().any():
    raise ValueError("Nama gambar kosong atau duplikat; periksa dataset.")

if not np.isfinite(X).all():
    raise ValueError("Fitur berisi nilai kosong atau tak hingga.")

if not np.isfinite(y).all():
    raise ValueError("Target berisi nilai kosong atau bukan angka.")

if np.unique(y).size < 2:
    raise ValueError("Target tidak memiliki variasi yang cukup untuk analisis.")

# Urutkan gambar berdasarkan timestamp pada nama file.
timestamp_teks = df[kolom_nama].str.extract(
    r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})",
    expand=False,
)

timestamp_gambar = pd.to_datetime(
    timestamp_teks,
    format="%Y-%m-%d_%H-%M-%S",
    errors="coerce",
)

if timestamp_gambar.isna().any():
    raise ValueError("Sebagian nama_file tidak memiliki timestamp yang dikenali.")

urutan = np.argsort(timestamp_gambar.to_numpy(), kind="stable")

X = X[urutan]
y = y[urutan]
df = df.iloc[urutan].reset_index(drop=True)

print(f"Jumlah sampel: {len(df)}")
print(f"Jumlah fitur awal: {len(kolom_fitur)}")
print(f"Rentang target: {y.min():,.0f} hingga {y.max():,.0f} sel/mL")
print("Pastikan target jumlah sel sudah cocok dengan timestamp gambar.")


# ============================================================
# 3. Pisahkan data secara acak: 80% train/pengembangan, 20% test
# ============================================================

jumlah_fold = 5
jumlah_fitur_pilih = min(50, X.shape[1])
ukuran_test = max(1, int(np.ceil(TEST_RATIO * len(y))))
semua_indeks = np.arange(len(y))
indeks_train, indeks_test = train_test_split(
    semua_indeks,
    test_size=ukuran_test,
    shuffle=True,
    random_state=SEED,
)

X_train = X[indeks_train]
y_train = y[indeks_train]
X_test = X[indeks_test]
y_test = y[indeks_test]
nama_file_test = df[kolom_nama].iloc[indeks_test].to_numpy()

if len(y_train) < jumlah_fold + 1:
    raise ValueError("Jumlah data train terlalu sedikit untuk CV 5-fold.")

if len(y_test) < 2:
    raise ValueError("Data test perlu minimal 2 sampel untuk menghitung R².")

cv = KFold(
    n_splits=jumlah_fold,
    shuffle=True,
    random_state=SEED,
)

print(f"Jumlah data train/pengembangan: {len(y_train)}")
print(f"Jumlah data test: {len(y_test)}")
print(f"Jumlah fold CV: {jumlah_fold}")
print(f"Fitur yang dipilih dalam setiap fold: {jumlah_fitur_pilih}")


# ============================================================
# 4. Definisikan pipeline dan lima model
# ============================================================

def buat_pipeline(estimator, perlu_scaling=False):
    langkah = [
        (
            "pilih_fitur",
            SelectKBest(
                score_func=f_regression,
                k=jumlah_fitur_pilih,
            ),
        )
    ]

    # KNN menggunakan jarak, sehingga fiturnya perlu distandardisasi.
    if perlu_scaling:
        langkah.append(("scaler", StandardScaler()))

    langkah.append(("model", estimator))
    return Pipeline(langkah)


model = {
    # DTS diartikan sebagai Decision Tree Regressor.
    "DTS": buat_pipeline(
        DecisionTreeRegressor(
            max_depth=5,
            min_samples_leaf=3,
            random_state=42,
        )
    ),
    "Random Forest": buat_pipeline(
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            max_features=0.5,
            random_state=42,
            n_jobs=-1,
        )
    ),
    "Gradient Boosting": buat_pipeline(
        GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=2,
            random_state=42,
        )
    ),
    "KNN": buat_pipeline(
        KNeighborsRegressor(
            n_neighbors=5,
            weights="distance",
            p=2,
        ),
        perlu_scaling=True,
    ),
    "XGBoost": buat_pipeline(
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
        )
    ),
}


# ============================================================
# 5. Random CV 5-fold pada 80% data untuk semua model
# ============================================================

metrik_cv = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

hasil_cv_per_fold = []
hasil_cv_ringkasan = []

for nama_model, estimator in model.items():
    print(f"Menjalankan CV: {nama_model}")

    skor = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring=metrik_cv,
        error_score="raise",
        n_jobs=1,
    )

    # Satu baris per model per fold, berisi MAE, RMSE, dan R².
    for nomor_fold in range(jumlah_fold):
        hasil_cv_per_fold.append(
            {
                "Model": nama_model,
                "Fold": nomor_fold + 1,
                "MAE": -skor["test_MAE"][nomor_fold],
                "RMSE": -skor["test_RMSE"][nomor_fold],
                "R2": skor["test_R2"][nomor_fold],
            }
        )

    # Ringkasan rata-rata dan standar deviasi skor antar-fold.
    hasil_cv_ringkasan.append(
        {
            "Model": nama_model,
            "MAE rata-rata": -skor["test_MAE"].mean(),
            "MAE std": skor["test_MAE"].std(),
            "RMSE rata-rata": -skor["test_RMSE"].mean(),
            "RMSE std": skor["test_RMSE"].std(),
            "R2 rata-rata": skor["test_R2"].mean(),
            "R2 std": skor["test_R2"].std(),
        }
    )

tabel_cv_per_fold = pd.DataFrame(hasil_cv_per_fold)

# Pemilihan model terbaik berdasarkan MAE CV rata-rata terendah.
tabel_cv_ringkasan = (
    pd.DataFrame(hasil_cv_ringkasan)
    .sort_values("MAE rata-rata")
    .reset_index(drop=True)
)

print("\nSkor CV setiap model dan fold:")
display(tabel_cv_per_fold.round(3))

print("\nRingkasan CV:")
display(tabel_cv_ringkasan.round(3))
print("MAE/RMSE lebih kecil lebih baik; R² lebih besar biasanya lebih baik.")


# ============================================================
# 6. Visualisasikan ringkasan CV kelima model
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

metrik_plot = [
    ("MAE rata-rata", "MAE rata-rata CV", "MAE std"),
    ("RMSE rata-rata", "RMSE rata-rata CV", "RMSE std"),
    ("R2 rata-rata", "R² rata-rata CV", "R2 std"),
]

for ax, (kolom_skor, judul, kolom_std) in zip(axes, metrik_plot):
    tabel_plot = tabel_cv_ringkasan.sort_values(kolom_skor)

    ax.bar(
        tabel_plot["Model"],
        tabel_plot[kolom_skor],
        yerr=tabel_plot[kolom_std],
        capsize=4,
    )
    ax.set_title(judul)
    ax.set_ylabel("Skor")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Perbandingan Lima Model — Random CV 5-Fold", fontsize=14)
fig.tight_layout()
plt.show()


# ============================================================
# 7. Latih ulang semua model pada seluruh data train 80%
#    lalu hitung skor train dan test untuk semua model
# ============================================================

def hitung_metrik(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }


hasil_train_test = []
prediksi_semua_test = {}

for nama_model, estimator in model.items():
    print(f"Melatih ulang pada 80% data random dan mengevaluasi: {nama_model}")

    # Model dilatih ulang pada seluruh 80% data train.
    estimator.fit(X_train, y_train)

    prediksi_train = estimator.predict(X_train)
    prediksi_test = estimator.predict(X_test)

    prediksi_semua_test[nama_model] = prediksi_test

    hasil_train_test.append(
        {
            "Model": nama_model,
            "Data": "Train",
            **hitung_metrik(y_train, prediksi_train),
        }
    )
    hasil_train_test.append(
        {
            "Model": nama_model,
            "Data": "Test",
            **hitung_metrik(y_test, prediksi_test),
        }
    )

tabel_train_test = pd.DataFrame(hasil_train_test)

print("\nEvaluasi semua model pada data train dan test:")
display(
    tabel_train_test
    .pivot(index="Model", columns="Data", values=["MAE", "RMSE", "R2"])
    .round(3)
)


# ============================================================
# 8. Visualisasikan residual dan Actual vs Predicted pada test
#    untuk semua model. Metrik berada di bawah label sumbu X.
# ============================================================

nama_model_semua = list(model.keys())
jumlah_model = len(nama_model_semua)

fig, axes = plt.subplots(
    jumlah_model,
    2,
    figsize=(16, 5.5 * jumlah_model),
    squeeze=False,
)

batas_min = min(
    y_test.min(),
    min(prediksi.min() for prediksi in prediksi_semua_test.values()),
)
batas_maks = max(
    y_test.max(),
    max(prediksi.max() for prediksi in prediksi_semua_test.values()),
)

for baris, nama_model in enumerate(nama_model_semua):
    prediksi_test = prediksi_semua_test[nama_model]
    residual_test = y_test - prediksi_test

    mae_test = mean_absolute_error(y_test, prediksi_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, prediksi_test))
    r2_test = r2_score(y_test, prediksi_test)

    # Residual — Data Test.
    axes[baris, 0].scatter(
        prediksi_test,
        residual_test,
        alpha=0.8,
    )
    axes[baris, 0].axhline(
        0,
        color="red",
        linestyle="--",
        label="Residual = 0",
    )
    axes[baris, 0].set_title(f"Residual — Data Test ({nama_model})")
    axes[baris, 0].set_xlabel("Predicted jumlah sel")
    axes[baris, 0].set_ylabel("Residual (Actual - Predicted)")
    axes[baris, 0].legend()
    axes[baris, 0].grid(True, alpha=0.3)

    # Actual vs Predicted — Data Test.
    axes[baris, 1].scatter(
        y_test,
        prediksi_test,
        alpha=0.8,
    )
    axes[baris, 1].plot(
        [batas_min, batas_maks],
        [batas_min, batas_maks],
        color="red",
        linestyle="--",
        label="Prediksi sempurna",
    )
    axes[baris, 1].set_title(
        f"Actual vs Predicted — Data Test ({nama_model})"
    )
    axes[baris, 1].set_xlabel("Actual jumlah sel")
    axes[baris, 1].set_ylabel("Predicted jumlah sel")
    axes[baris, 1].legend()
    axes[baris, 1].grid(True, alpha=0.3)

    # Metrik test ditampilkan tepat di bawah label sumbu X,
    # di luar area plot dan tanpa judul "Evaluasi test".
    teks_metrik = (
        f"MAE = {mae_test:,.0f}   |   "
        f"RMSE = {rmse_test:,.0f}   |   "
        f"R² = {r2_test:.3f}"
    )
    axes[baris, 1].text(
        0.5,
        -0.32,
        teks_metrik,
        transform=axes[baris, 1].transAxes,
        ha="center",
        va="top",
    )

# Ruang vertikal antarbari disediakan agar teks metrik tidak bertumpuk.
fig.subplots_adjust(
    left=0.08,
    right=0.98,
    top=0.97,
    bottom=0.04,
    hspace=0.9,
    wspace=0.28,
)
plt.show()


# ============================================================
# 9. Tentukan model pilihan berdasarkan hasil CV
# ============================================================

nama_model_terbaik = tabel_cv_ringkasan.loc[0, "Model"]
model_terbaik = model[nama_model_terbaik]

print(f"Model terbaik berdasarkan MAE rata-rata CV: {nama_model_terbaik}")


# ============================================================
# 10. Simpan tabel hasil, prediksi test, dan model terpilih
# ============================================================

file_cv_per_fold = folder_output / "hasil_cv_per_fold.csv"
file_cv_ringkasan = folder_output / "hasil_cv_ringkasan.csv"
file_train_test = folder_output / "hasil_evaluasi_train_test_semua_model.csv"
file_prediksi_test = folder_output / "prediksi_test_semua_model.csv"
file_model_terbaik = folder_output / "model_terbaik.joblib"

tabel_cv_per_fold.to_csv(
    file_cv_per_fold,
    index=False,
    encoding="utf-8-sig",
)

tabel_cv_ringkasan.to_csv(
    file_cv_ringkasan,
    index=False,
    encoding="utf-8-sig",
)

tabel_train_test.to_csv(
    file_train_test,
    index=False,
    encoding="utf-8-sig",
)

data_prediksi_test = pd.DataFrame(
    {
        "nama_file": nama_file_test,
        "actual": y_test,
    }
)

for nama_model, prediksi in prediksi_semua_test.items():
    data_prediksi_test[f"predicted_{nama_model}"] = prediksi
    data_prediksi_test[f"residual_{nama_model}"] = y_test - prediksi

data_prediksi_test.to_csv(
    file_prediksi_test,
    index=False,
    encoding="utf-8-sig",
)

# Model terpilih sudah dilatih ulang pada seluruh data train 80%.
joblib.dump(model_terbaik, file_model_terbaik)

print("\nFile hasil disimpan:")
print(f"- Skor CV tiap fold: {file_cv_per_fold}")
print(f"- Ringkasan CV: {file_cv_ringkasan}")
print(f"- Evaluasi train/test semua model: {file_train_test}")
print(f"- Prediksi dan residual test semua model: {file_prediksi_test}")
print(f"- Model terbaik berdasarkan CV: {file_model_terbaik}")